# Python Core Mechanics

Five foundational Python patterns — applied to trading and finance contexts.
Each section includes explanation, annotated code, and a mini challenge.

| # | Mechanic | Why it matters |
|---|----------|----------------|
| 1 | Loops + `enumerate` | Avoid manual index bugs, iterate with position |
| 2 | Dictionaries | O(1) lookup — core of frequency maps, grouping, caching |
| 3 | Sorting | `key=` unlocks sorting anything by any field |
| 4 | List comprehensions | Replace map/filter with readable one-liners |
| 5 | Strings | Parse, format, and split structured text data |

---

## Mechanic 1: Loops + `enumerate`

`enumerate(iterable)` yields `(index, value)` pairs — you never need a manual counter variable.

```python
# Without enumerate (fragile)
i = 0
for item in items:
    print(i, item)
    i += 1

# With enumerate (clean)
for i, item in enumerate(items):
    print(i, item)
```

You can also pass `start=` to offset the index — useful for 1-indexed output.

In [ ]:
prices = [101.5, 98.2, 103.7, 99.1, 105.0]

# Basic enumerate
print("Bar data (0-indexed):")
for i, price in enumerate(prices):
    print(f"  Bar {i}: ${price}")

print()

# Offset start for 1-indexed candle labels
print("Candle labels (1-indexed):")
for i, price in enumerate(prices, start=1):
    print(f"  Candle {i}: ${price}")

In [ ]:
# Mini challenge: flag bars where price crossed above $100
print("Bars above $100 (with position):")
for i, price in enumerate(prices):
    if price > 100:
        status = '▲ ABOVE' if price > 100 else '▼ BELOW'
        print(f"  Bar {i}: ${price}  {status}")

# Practical: detect where price drops below previous bar
print("\nDownbars:")
for i, price in enumerate(prices[1:], start=1):
    if price < prices[i - 1]:
        print(f"  Bar {i}: ${price} < Bar {i-1}: ${prices[i-1]}  ↓")

## Mechanic 2: Dictionaries

Dictionaries are Python's **hash map** — O(1) average insert and lookup.

Key patterns:
- `.get(key, default)` — safe access without KeyError
- `.items()` — iterate key-value pairs
- Counting pattern: `d[k] = d.get(k, 0) + 1`
- Grouping pattern: `d.setdefault(k, []).append(v)`

In [ ]:
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}

# Iterate with .items()
print("Current portfolio:")
for ticker, info in portfolio.items():
    print(f"  {ticker}: {info['qty']} contracts @ {info['entry']:,}")

# Safe access — no KeyError if key is missing
pnl = {"NQ": 1500, "ES": -200}
print()
for ticker in portfolio:
    print(f"  {ticker} P&L: ${pnl.get(ticker, 0):,}")

In [ ]:
# Counting pattern — frequency map
trades = ["BUY", "SELL", "BUY", "BUY", "SELL", "BUY", "SELL", "SELL", "SELL"]
count = {}
for t in trades:
    count[t] = count.get(t, 0) + 1
print(f"Trade counts: {count}")
win_rate = count.get('BUY', 0) / len(trades)
print(f"Buy rate: {win_rate:.1%}")

# Grouping pattern — group trades by direction
trade_log = [
    {"ticker": "NQ", "dir": "BUY",  "pnl": 500},
    {"ticker": "ES", "dir": "SELL", "pnl": -200},
    {"ticker": "NQ", "dir": "BUY",  "pnl": 800},
    {"ticker": "GC", "dir": "BUY",  "pnl": -100},
    {"ticker": "ES", "dir": "SELL", "pnl": 300},
]
by_ticker = {}
for trade in trade_log:
    by_ticker.setdefault(trade['ticker'], []).append(trade['pnl'])

print("\nP&L by ticker:")
for ticker, pnls in by_ticker.items():
    print(f"  {ticker}: total=${sum(pnls):,}  trades={len(pnls)}")

## Mechanic 3: Sorting

- `sorted(iterable)` — returns a **new** sorted list
- `list.sort()` — sorts **in place**, returns None
- `key=` accepts any callable — use `lambda` for inline logic
- `reverse=True` for descending order

```python
sorted(items, key=lambda x: x['field'], reverse=True)
```

In [ ]:
# Basic sorts
closes = [103.2, 98.5, 101.0, 99.8, 105.3, 97.1]
print(f"Original:   {closes}")
print(f"Ascending:  {sorted(closes)}")
print(f"Descending: {sorted(closes, reverse=True)}")

# Sort list of dicts by field
signals = [
    {"ticker": "NQ", "score": 0.82, "direction": "LONG"},
    {"ticker": "ES", "score": 0.65, "direction": "SHORT"},
    {"ticker": "GC", "score": 0.91, "direction": "LONG"},
    {"ticker": "ZB", "score": 0.44, "direction": "SHORT"},
]
ranked = sorted(signals, key=lambda s: s['score'], reverse=True)
print("\nSignals ranked by score:")
for rank, s in enumerate(ranked, 1):
    print(f"  #{rank} {s['ticker']} ({s['direction']}) — score {s['score']}")

In [ ]:
# Multi-key sort: primary by direction, secondary by score descending
multi_sorted = sorted(signals, key=lambda s: (s['direction'], -s['score']))
print("Sorted by direction then score desc:")
for s in multi_sorted:
    print(f"  {s['direction']:<6} {s['ticker']}  score={s['score']}")

# Mini challenge: sort portfolio by total notional value (qty * entry)
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}
by_notional = sorted(
    portfolio.items(),
    key=lambda x: x[1]['qty'] * x[1]['entry'],
    reverse=True
)
print("\nPortfolio by notional value:")
for ticker, info in by_notional:
    notional = info['qty'] * info['entry']
    print(f"  {ticker}: {info['qty']} x {info['entry']:,} = ${notional:,}")

## Mechanic 4: List Comprehensions

Compact syntax for building lists — replaces most `for` loops that append to a list.

```python
[expression for item in iterable if condition]
```

- The `if condition` part is optional
- You can nest comprehensions for 2D structures
- Dict comprehensions: `{k: v for k, v in ...}`
- Set comprehensions: `{x for x in ...}`

In [ ]:
# Basic: squares
squares = [x**2 for x in range(1, 11)]
print(f"Squares: {squares}")

# Filtered
even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
print(f"Even squares: {even_squares}")

# Applied to price data: compute bar-over-bar returns
prices = [101.5, 98.2, 103.7, 99.1, 105.0]
returns = [(prices[i] - prices[i-1]) / prices[i-1] for i in range(1, len(prices))]
print(f"\nReturns: {[round(r, 4) for r in returns]}")
print(f"Avg return: {sum(returns)/len(returns):.4f}")
print(f"Down bars:  {sum(1 for r in returns if r < 0)} of {len(returns)}")

In [ ]:
# Dict comprehension: ticker → notional value
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}
notional = {t: info['qty'] * info['entry'] for t, info in portfolio.items()}
print(f"Notional values: {notional}")

# Nested comprehension: all (i,j) trade pairs for correlation analysis
tickers = ['NQ', 'ES', 'GC']
pairs = [(a, b) for i, a in enumerate(tickers) for b in tickers[i+1:]]
print(f"\nTicker pairs for correlation: {pairs}")

# Mini challenge: filter signals with score > 0.7 and direction LONG
signals = [
    {"ticker": "NQ", "score": 0.82, "direction": "LONG"},
    {"ticker": "ES", "score": 0.65, "direction": "SHORT"},
    {"ticker": "GC", "score": 0.91, "direction": "LONG"},
    {"ticker": "ZB", "score": 0.44, "direction": "SHORT"},
]
qualified = [s['ticker'] for s in signals if s['score'] > 0.70 and s['direction'] == 'LONG']
print(f"\nQualified LONG signals (score > 0.70): {qualified}")

## Mechanic 5: Strings

Strings are **immutable sequences**. Key methods:

| Method | What it does |
|--------|-------------|
| `.strip()` | Remove leading/trailing whitespace |
| `.split(sep)` | Split into list on separator |
| `sep.join(list)` | Join list back into string |
| `.upper()` / `.lower()` | Case conversion |
| `.startswith()` / `.endswith()` | Prefix/suffix check |
| `.replace(old, new)` | Substitute substring |
| `f"{value:.2f}"` | Formatted string literals |


In [ ]:
# Clean and normalize raw input
raw = "  NQ Futures  "
clean = raw.strip()
print(f"Stripped: '{clean}'")
print(f"Upper:    '{clean.upper()}'")
print(f"Lower:    '{clean.lower()}'")

# Parse CSV trade data
csv_row = "NQ,2,19800,LONG,2024-01-15"
ticker, qty, entry, direction, date = csv_row.split(",")
print(f"\nParsed: ticker={ticker} qty={qty} entry={entry} dir={direction} date={date}")

# Rejoin with different separator
print(f"Formatted: {' | '.join([ticker, qty, entry, direction])}")

In [ ]:
# f-strings with format specs
entry = 19842.50
stop  = 19750.00
risk  = entry - stop
size  = 2
dollar_risk = risk * size * 20  # NQ point value = $20

print(f"Entry:        {entry:>12,.2f}")
print(f"Stop:         {stop:>12,.2f}")
print(f"Risk (pts):   {risk:>12.2f}")
print(f"Dollar risk:  {dollar_risk:>12,.2f}")

# Mini challenge: parse 'TICKER_DIRECTION' tags into structured dict
tags = ["LONG_NQ", "SHORT_ES", "LONG_GC", "FLAT_ZB"]
parsed = {}
for tag in tags:
    direction, ticker = tag.split("_", 1)
    parsed[ticker] = direction
print(f"\nParsed tags: {parsed}")

# Filter to only active (non-FLAT) positions
active = {t: d for t, d in parsed.items() if d != 'FLAT'}
print(f"Active positions: {active}")

---
## Quick Reference

```python
# enumerate
for i, item in enumerate(lst, start=1): ...

# dict safe access + counting
d.get(key, default)
d[k] = d.get(k, 0) + 1
d.setdefault(k, []).append(v)

# sorting by field
sorted(items, key=lambda x: x['field'], reverse=True)

# comprehension with filter
[f(x) for x in lst if condition(x)]

# string split/join
parts = s.split(',')   # str → list
s = ','.join(parts)    # list → str
```